# HL-mesh lab

Hexagonal L-meshes (HL-meshes) from the planar hexagonal meshes of PH-CPF (Pluta et al. 2021)
in `data/ph_results/`. Spheres on the mesh vertices, oriented planes on the hexagons.

- `hl_mesh_computation(name, tau)`: sphere radius = τ × smallest curvature-sphere radius,
  optimisation (oriented contact, strip and face fairness, proximity to the reference
  surface), `improve_contact()`, export to `notebooks/out/lmesh/HL_results/<mesh>/exp_<τ>/`.
- `hl_mesh_simple(name, tau)`: the same without a reference surface.
- Batch sweep: every mesh at several τ, with LaTeX tables and images.

Choose the mesh and τ in the last lines of the setup cell.

**Viewer:** `kv.init()` starts a kayviz viewer and prints its URL; open it in a browser. To share one viewer between several notebooks, run `python -m kayviz` in a terminal first.

## Python paths and configurations

data path: `data/ph_results`

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

from LMesh.lmesh_serializers import lmesh_visualize
import kayviz as kv
import pymeshlab
from LMesh.L_mesh import Lmesh
from hanan.geometry import Mesh
from hanan.optimization.optimizer import Optimizer
from hanan.optimization.laplacian_fairness import LaplacianFairness
from hanan.geometry.utils import (
    read_obj,
    read_off,
    normalize_vertices,
    compute_cone_axes,
    conical_mesh_smallest_curvature_sphere
)
import numpy as np
import igl

from hanan.optimization.optimizer import Optimizer
from LMesh.Optimization.Or_contactFree import Or_contactFree
from LMesh.Optimization.HLmesh_reg_strips import HLmesh_reg_strips
from LMesh.Optimization.HLmesh_reg_hex_faces import HLmesh_reg_hex_faces
from LMesh.Optimization.Tangential_Pos import TDPositive
from LMesh.Optimization.Tangential_Dist import Tangential_Dist
from LMesh.Optimization.ContactPointsProximity import ContactPointsProximity
from LMesh.Optimization.ContactPointsGlide import ContactPointsGlide

DATA     = os.path.join(PROJECT_ROOT, "data", "THlmeshes_input","PH")
OUT_PATH = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "HL_results")


def boundary_adjacency(top):
    """Boundary vertex -> its two neighbours along the boundary loop."""
    idx, adj = [], []
    for loop in top.boundary_curves():
        n = len(loop)
        for k, v in enumerate(loop):
            idx.append(v)
            adj.append([loop[k - 1], loop[(k + 1) % n]])
    return idx, adj


def add_boundary_fairness(opt, top, w_c, w_r, damp_factor=1.0, damp_iteration=1000):
    """Laplacian fairness of sphere centres and radii along the boundary curves only.

    opt.set_fairness() cannot be used here: it takes the position in adj_list as
    the vertex index, so skipping inner vertices with empty lists would pull them
    to the origin.
    """
    idx, adj = boundary_adjacency(top)
    for var, dim, w in (("cv", 3, w_c), ("rv", 1, w_r)):
        term = LaplacianFairness()
        term._initialize_objective(opt.X, opt.var_idx, var, idx, adj, dim, damp_factor, damp_iteration)
        term.name = "bd_fairness_" + var
        term.w = w
        opt.objective_terms[term.name] = term


def interior_mean_radii(mesh, radii):
    """Boundary radii <- mean radius of the interior 1-ring neighbours.

    Vertices with no interior neighbour (e.g. corners) take the mean of their
    already-set boundary neighbours, repeated until every one is set.
    """
    radii = radii.copy()
    adj = mesh.vertex_adjacency_list()
    bd = set(mesh.boundary_vertices().tolist())
    todo = []
    for v in bd:
        inner = [u for u in adj[v] if u not in bd]
        if inner:
            radii[v] = radii[inner].mean()
        else:
            todo.append(v)
    done = bd - set(todo)
    while todo:
        left = []
        for v in todo:
            nb = [u for u in adj[v] if u in done]
            if nb:
                radii[v] = radii[nb].mean()
            else:
                left.append(v)
        if len(left) == len(todo):
            break  # isolated boundary component: keep the curvature radii
        done |= set(todo) - set(left)
        todo = left
    return radii


def setup_optimizer(lmesh, Vr=None, Fr=None, w_bd=0.0):
    opt = Optimizer()

    weights = {
            "Or_Contact": 2.0, 
            #"TD": 0,
            #"TDpos": 0,
            "HLmesh_Reg_Strips": 5e-3,
            "HLmesh_Reg_Hex_Faces": 5e-3, 
             "contactProximity": 5e-4,
             "contactGliding": 5e-4,
            #"contactProximity": 0,
            #"contactGliding": 0,
            #"npf_fairness": 0    
            }

    cf  = lmesh.top.vertices[:, :3]
    rf  = lmesh.top.vertices[:, 3]
    npf = lmesh.PF[:, :3]
    hpf = lmesh.PF[:, 3]

    # npf/hpf are per-FACE variables (one oriented plane per L-mesh face),
    # so fairness must use face->face adjacency.  A vertex adjacency list
    # would index past the end of the face array.
    plane_adj = lmesh.top.face_face_adjacency_list()
    si_sj_e = lmesh.top.edge_vertices()  # edge->(start_vertex, end_vertex) pairs

    opt.add_variable("cv", cf.flatten())
    opt.add_variable("rv", rf)
    opt.add_variable("npf", npf.flatten())
    opt.add_variable("hpf", hpf.flatten())

    # Oriented contact term
    Or_contact = Or_contactFree()
    Or_contact.name = "or_contact"
    opt.add_objective_term(
        Or_contact,
        args=(lmesh.top,),
        w=weights["Or_Contact"],
        ce=1,
    )

    # Regularity of strips
    HLmesh_Reg_Strips = HLmesh_reg_strips()
    HLmesh_Reg_Strips.name = "HLmesh_Reg_Strips"
    opt.add_objective_term(
        HLmesh_Reg_Strips,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Strips"],
        ce=0,
    )

    # Tangential_Dist_term = Tangential_Dist()
    # Tangential_Dist_term.name = "tangential_dist"
    # opt.add_objective_term(
    #     Tangential_Dist_term,
    #     args=(si_sj_e, ),
    #     w=weights["TD"],
    #     ce=0,
    # )

    # Tangential_Dist_pos_term = Tangential_Dist()
    # Tangential_Dist_pos_term.name = "tangential_dist_pos"
    # opt.add_objective_term(
    #     Tangential_Dist_pos_term,
    #     args=(si_sj_e, ),
    #     w=weights["TDpos"],
    #     ce=0,
    # )


    # Regularity of hex faces
    HLmesh_Reg_Hex_Faces = HLmesh_reg_hex_faces()
    HLmesh_Reg_Hex_Faces.name = "HLmesh_Reg_Hex_Faces"
    opt.add_objective_term(
        HLmesh_Reg_Hex_Faces,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Hex_Faces"],
        ce=0,
    )

    if Vr is not None and Fr is not None:

        ContactPointsProximity_term = ContactPointsProximity()
        ContactPointsProximity_term.name = "contactProximity"
        opt.add_objective_term(
            ContactPointsProximity_term,
            args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
            w=weights["contactProximity"],
            ce=1,
        )
            
        ContactPointsGlide_term = ContactPointsGlide()
        ContactPointsGlide_term.name = "contactGliding"
        opt.add_objective_term(
            ContactPointsGlide_term,
            args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
            w=weights["contactGliding"],
            ce=1,
        )

    # Normal vector fairness
    #opt.set_fairness("npf", plane_adj, 3, 0.1, 10, weights["npf_fairness"])
    # Boundary fairness of sphere centres and radii (off when w_bd = 0)
    if w_bd > 0:
        add_boundary_fairness(opt, lmesh.top, w_bd, w_bd)

    opt.unitize_variable("npf", 3, 10)
    #opt.prev_iteration_regularizer(w=1e-4)
    opt.initialize_optimizer(verbose=True, adaptive_mu=False)


    return opt

def ref_surface_mesh(V, F):

  
    # Refine mesh for proximity 
    m  = pymeshlab.Mesh(vertex_matrix=V, face_matrix=F)
    ms = pymeshlab.MeshSet()
    ms.add_mesh(m)

    ms.apply_filter("meshing_isotropic_explicit_remeshing", 
                    targetlen=pymeshlab.PercentageValue(1.0),  # 1% of bbox diagonal
                    iterations=5)

    ms.apply_filter("apply_coord_taubin_smoothing", lambda_=0.5, mu=-0.53, stepsmoothnum=10)

    # Get the current mesh object
    mesh = ms.current_mesh()

    # Extract as numpy arrays
    Vr = mesh.vertex_matrix()       # (N, 3) float64 — vertex positions
    Fr = mesh.face_matrix()

    return Vr, Fr   

def hl_mesh_from_tri(name, tau):
    output_path = os.path.join(OUT_PATH, name)
    
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    print("Reading mesh:", name)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)

    F = [f for f in F if len(f) == 3]  # Filter out non-triangular faces

    F = np.array(F, dtype=np.int32)
    V = np.array(V, dtype=np.float64)

    kv.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()

    basemesh.make_mesh(V, F)

    _, _, k1, k2, _ = igl.principal_curvature(V, F)
    min_curvature = np.maximum( np.abs(k1), np.abs(k2))
    smallest_radii = 1.0 / np.abs(min_curvature)
    radii =  tau * smallest_radii

    normals = basemesh.vertex_normals
    inner_v = basemesh.inner_vertices()

    orplanes = np.hstack((normals, -np.sum(normals * V, axis=1)[:, np.newaxis]))

    barycenters  = basemesh.face_barycenters()
    bary_normals = np.mean(normals[basemesh.faces], axis=1)

    # unitize bary_normals
    bary_normals /= np.linalg.norm(bary_normals, axis=1)[:, np.newaxis]
    
    bary_radii = radii[basemesh.faces].mean(axis=1)
    centers = barycenters + bary_radii[:, np.newaxis] * bary_normals

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="dual",
        mode="OrPlanes",
        planes=orplanes,
        radii=radii,
        centers=centers,
        
    )

    experiment_name = f"{name}_tau_{tau:.2f}"
    #lmesh.improve_contact()
    Vs, Fs = lmesh.sphere_centers_mesh()
    lmesh_visualize(lmesh)

    # lmesh.export_data(
    #     output_path, experiment_name )

    opt = setup_optimizer(lmesh)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
    
        
    # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)
    
    # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)


    lmesh.improve_contact()

    lmesh_visualize(lmesh) 
        
        # sC, sF = lmesh.sphere_center_mesh()
        # kv.register_surface_mesh("Sphere Centers", sC, sF, color=(0.7, 0.9, 0.7), show_edges=True)

def orient_convex_side(V, F):
    """Flip the face winding so the normals point to the convex side (majority vote).

    The cone axes follow the face normals (cone_axis_from_planes), so this decides
    on which side the spheres are placed. Where K > 0 the umbrella vector
    (neighbour average - vertex) points to the concave side; works for open patches.
    """
    vn = np.zeros_like(V)
    nb = [set() for _ in V]
    for f in F:
        P = V[f]
        n = np.sum(np.cross(P, np.roll(P, -1, 0)), 0)
        vn[f] += n / np.linalg.norm(n)
        for i, a in enumerate(f):
            nb[a].update([f[i - 1], f[(i + 1) % len(f)]])
    s = [vn[i] @ (V[list(nb[i])].mean(0) - V[i]) for i in range(len(V)) if nb[i]]
    if np.mean(np.array(s) > 0) > 0.5:
        F = [list(f)[::-1] for f in F]
        print("    [orient] flipped normals to the convex side")
    return F

def hl_mesh_computation(name, tau, bd_fairness=False, w_bd=1e-3, bd_radii="interior", n_steps=20,
                        fix_bd_radii=False):
    """Build and optimise one HL-mesh, the same way as the batch run_case.

    Args:
        name:        Mesh name in DATA (without .obj).
        tau:         Initial radii = tau * smallest curvature-sphere radius.
        bd_fairness: Add Laplacian fairness of the boundary sphere centres and
                     radii along the boundary curves.
        w_bd:        Weight of that fairness term (centres and radii).
        bd_radii:    Initial radii of the boundary spheres: "curvature" (same rule
                     as the interior), "interior" (mean of the interior
                     neighbours) or "zero".
        n_steps:     Optimisation steps.
        fix_bd_radii: Hold the boundary radii constant at their initial value,
                     in the optimisation and in improve_contact().

    Exports to OUT_PATH/[name]/exp_[tau][_bdfair][_bdr<bd_radii> | _bdfixed]/,
    plus the reference surface used by the proximity/glide terms.
    """
    tag = f"exp_{tau:.2f}" + ("_bdfair" if bd_fairness else "")
    if fix_bd_radii:
        tag += "_bdfixed"
    elif bd_radii != "curvature":
        tag += "_bdr" + bd_radii
    exp_dir = os.path.join(OUT_PATH, name, tag)
    os.makedirs(exp_dir, exist_ok=True)

    print("Reading mesh:", name)
    V, F = load_oriented(name, DATA)

    #V = normalize_vertices(V, 2)

    Vr, Fr = ref_pair(name, DATA)

    kv.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)
    kv.register_surface_mesh("Reference Mesh", Vr, Fr, color=(0.1, 0.0, 0.0), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )

    kv.add_vector_quantity("Input Mesh", "cone_axes", axes)
    kv.add_vector_quantity("Input Mesh", "face_normals", basemesh.face_normals, defined_on='faces')

    scaled_axes = axes / np.sin(angles[:, None])
    radii = tau * smallest_radii
    if bd_radii == "interior":
        radii = interior_mean_radii(basemesh, radii)
    elif bd_radii == "zero":
        radii[basemesh.boundary_vertices()] = 0.0
    centers = basemesh.vertices + radii[:, None] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    lmesh.export_data(exp_dir, "init")
    with open(os.path.join(exp_dir, "reference_surface.obj"), "w") as fh:
        fh.writelines(f"v {x:.10g} {y:.10g} {z:.10g}\n" for x, y, z in Vr)
        fh.writelines(f"f {i+1} {j+1} {k+1}\n" for i, j, k in Fr)

    lmesh_visualize(lmesh)

    opt = setup_optimizer(lmesh, Vr, Fr, w_bd=w_bd if bd_fairness else 0.0)
    bd_vertices = basemesh.boundary_vertices()
    if fix_bd_radii:
        opt.fix_variables("rv", bd_vertices)

    for _ in range(n_steps):
        opt.get_gradients()
        opt.optimize_step()

    lmesh.update_spheres_planes(
        opt.unpack("cv").reshape(-1, 3), opt.unpack("rv"),
        opt.unpack("npf").reshape(-1, 3), opt.unpack("hpf"),
    )

    opt.print_report()
    if fix_bd_radii:
        # Same as lmesh.improve_contact(), with the boundary radii held fixed:
        # its optimiser starts with every radius free.
        lmesh.setup_optimization()
        lmesh.Optimizer.fix_variables("rv", bd_vertices)
        for _ in range(20):
            lmesh.Optimizer.get_gradients()
            lmesh.Optimizer.optimize_step()
        lmesh._apply_optimizer_result()
        lmesh.compute_lmesh_structure()
    else:
        lmesh.improve_contact()
    print(f"Oriented contact deviation: {lmesh.compute_oriented_contact_dev():.3e}")

    # improve_contact() ends with compute_lmesh_structure(), so the improved
    # state is fully rebuilt here and safe to export.
    lmesh.export_data(exp_dir, "lmesh")
    print("Exported to", exp_dir)

    lmesh_visualize(lmesh)
    return lmesh


def hl_mesh_fixed_boundary(name, tau, bd_fairness=False, w_bd=1e-3, n_steps=20):
    """hl_mesh_computation with the boundary spheres fixed at radius 0.

    The boundary centres stay free (a radius-0 sphere is a point on its planes),
    so bd_fairness still smooths them along the boundary curves.
    Exports to OUT_PATH/[name]/exp_[tau][_bdfair]_bdfixed/.
    """
    return hl_mesh_computation(name, tau, bd_fairness=bd_fairness, w_bd=w_bd,
                               bd_radii="zero", n_steps=n_steps, fix_bd_radii=True)


def _signed_volume(V, faces):
    vol = 0.0
    for f in faces:
        f = list(f)
        for k in range(1, len(f) - 1):
            vol += np.dot(V[f[0]], np.cross(V[f[k]], V[f[k + 1]])) / 6.0
    return vol


def load_oriented(name, data_dir):
    """Read a mesh and return (V, F) with consistently inward-pointing normals.

    F stays a list of index lists: some meshes mix hexagons with pentagons and
    heptagons, which do not fit in one numpy array.
    """
    V, F = read_obj(os.path.join(data_dir, name + ".obj"))
    #V = normalize_vertices(V, 2)
    F = [list(f) for f in F]
    if _signed_volume(V, F) > 0:
        F = [f[::-1] for f in F]     # reverse winding -> flip normals
        print(f"    [orient] flipped normals for {name}")
    return V, F


_REF_CACHE = {}

def ref_pair(name, data_dir):
    """Remeshed reference surface for the proximity/glide terms.

    setup_optimizer registers ContactPointsProximity / ContactPointsGlide
    unconditionally (at weight 0), and both need a real (Vr, Fr), so build the
    same reference hl_mesh_computation uses.  Cached: it is the same surface for
    every tau of a given mesh.
    """
    if name in _REF_CACHE:
        return _REF_CACHE[name]
    ref_name = name.replace(".out_d_p", "")
    Vr, Fr = read_obj(os.path.join(data_dir, ref_name + ".obj"))
    # Meshes without a triangulated ".out_d_p" source are their own reference;
    # pymeshlab needs triangles, so fan-triangulate their polygons.
    Fr = np.array([[f[0], f[k], f[k + 1]] for f in Fr for k in range(1, len(f) - 1)])
    #Vr, Fr = ref_surface_mesh(Vr, Fr)
    #Vr = normalize_vertices(Vr, 2)
    _REF_CACHE[name] = (Vr, Fr)
    return _REF_CACHE[name]


def hl_mesh_simple(name, tau):

    exp_dir = os.path.join(OUT_PATH, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)
    

    print("Reading mesh:", name)

    ref_name = name.replace(".out_d_p", ".obj")

    if ".out_d_p" in name:
        ref_name = name.replace(".out_d_p", ".obj")
    else:
        ref_name = name + ".obj"
    
    kv.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    # kv.register_surface_mesh("Reference Mesh", Vr, Fr, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )

    face_normals = basemesh.face_normals

    kv.add_vector_quantity("Input Mesh", "cone_axes", axes)
    kv.add_vector_quantity("Input Mesh", "face_normals", face_normals, defined_on='faces')

    scaled_axes = axes / np.sin(angles[:, None])

    radii   = tau * smallest_radii

    centers = V + radii[:, np.newaxis] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    experiment_name = f"{name}_tau_{tau :.2f}"
    lmesh.export_data(exp_dir, "init")
    
        
    
        # improve_contact() ends with compute_lmesh_structure(), so the improved
        # state is fully rebuilt here and safe to export.
      
    lmesh_visualize(lmesh)

    # lmesh.export_data(
    #     output_path, experiment_name )

    opt = setup_optimizer(lmesh)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
        # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)

    # # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)

    opt.print_report()
    lmesh.improve_contact()

    lmesh.export_data(exp_dir, "lmesh")
    
    # kv.register_surface_mesh("Gauss L-mesh", onormals, lmesh.top.dual_top(), color=(0.8, 0.8, 0.8), opacity=0.1, show_edges=True, edge_color=(1.0, 0.5, 0.1), edge_radius = 0.002)
    
    lmesh_visualize(lmesh) 
        

kv.init()   # pings the server; raises if it is not running

ph_mesh_names = [
    "bumpy_patch.out_d_p",
    "coiled_torus_segment.out_d_p",
    "flared_nautilus.out_d_p",
    "helical_superellipsoid_tube.out_d_p",
    "hexMeshTest",
    "loft_00014.out_d_p",
    "nautilus.out_d_p",
    "ribbed_conch.out_d_p",
    "seashell.out_d_p",
    "shell_00001.out_d_p",
    "shell_00002.out_d_p",
    "shell_nautilus.out_d_p",
    "testSimpleOptimized.out_d_p",
    "tube.out_d_p",
    "turritella_shell.out_d_p"
]

#ph_input_seashell_ex6.out_d_
kv.clear()
#hl_mesh_simple(ph_mesh_names[6], 0.8)
hl_mesh_computation(ph_mesh_names[3], 0.8, bd_fairness=False, w_bd=1e-3, bd_radii="interior", n_steps=20)
#kv.show()

Project root: /home/anthony/KAUSTLocal/Laguerre_project
[kayviz] Viewer running at http://127.0.0.1:8001
Reading mesh: helical_superellipsoid_tube.out_d_p
Mesh |V|=1310, |F|=553, |E|=1862
Mesh |V|=1310, |F|=553, |E|=1862
L-mesh exported to folder: /home/anthony/KAUSTLocal/Laguerre_project/notebooks/out/lmesh/HL_results/helical_superellipsoid_tube.out_d_p/exp_0.80_bdrinterior


ERROR:    keepalive ping failed
Traceback (most recent call last):
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py", line 1233, in keepalive_ping
    pong_waiter = await self.ping()
                  ^^^^^^^^^^^^^^^^^
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py", line 843, in ping
    await self.write_frame(True, OP_PING, data)
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py", line 1184, in write_frame
    await self.drain()
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py", line 1167, in drain
    await self._drain()
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py", line 328, in _drain
    await self._drain_helper()
  File "/home/anthony/anaconda3/envs/Lproj/lib/python3.11/site-packages/websockets/legacy/protocol.py"

Thread pool: 6 workers (terms=6, cpus=32)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 7.115770e-03  | 2.392040e-07  | 2.092088e+00  | 4.590605e-05  | N/A      | 0.4277  
10     | 3.633231e-06  | 7.144661e-10  | 4.273119e-02  | 4.590605e-05  | N/A      | 0.0779  
20     | 1.799172e-06  | 3.612797e-10  | 1.963668e-02  | 4.590605e-05  | N/A      | 0.0721  

 WEIGHTS REPORT
Term                        Weight
----------------------------------
or_contact              2.0000e+00
HLmesh_Reg_Strips       5.0000e-09
HLmesh_Reg_Hex_Faces    5.0000e-09
contactProximity        5.0000e-04
contactGliding          5.0000e-04
npf_unit                1.0000e+01
 Total Iterations: 20
 Time per Iteration: 0.2051 seconds
 ENERGY REPORT
Term                      Energy      Normalized  Contrib.
------------------------

## Batch sweep — energies, contact deviation, LaTeX tables, view grids

Runs every mesh in `ph_mesh_names` at `tau in [0.5, 0.7, 0.85]`.
Writes one LaTeX table and one image grid per tau under `out/`.

Needs an open viewer tab for the screenshots (`kv.show()`).


In [7]:
from LMesh.lmesh_serializers import lmesh_visualize
# ── Batch sweep: every mesh x every tau, with LaTeX tables and view grids ──────
#
# For each (mesh, tau):
#   * build the L-mesh, run 10 optimisation steps
#   * record the final optimisation energy (total and per term)
#   * run improve_contact() and record the oriented contact deviation
#   * record the weights actually used
#   * capture two views of the scene
#
# Outputs, one set per tau value:
#   out/tables/ph_energy_tau[TAU].tex        -- LaTeX table, ready to \input
#   out/images/ph_images/grid_tau[TAU].png   -- one row per mesh, two views
#   out/images/ph_images/[mesh]_[tau].png    -- the individual views

import os, json, io, re, contextlib
import numpy as np

TAUS       = [0.5, 0.75, 0.85]
N_STEPS    = 20

# Boundary spheres, as in hl_mesh_computation:
#   BD_RADII     initial boundary radii: "curvature" | "interior" | "zero"
#   BD_FAIRNESS  Laplacian fairness of boundary centres/radii (weight W_BD)
BD_RADII    = "interior"
BD_FAIRNESS = False
W_BD        = 1e-3
# Suffix for the export folders, tables and images of this configuration
TAG = ("_bdfair" if BD_FAIRNESS else "") + ("" if BD_RADII == "curvature" else "_bdr" + BD_RADII)

# ── Framing for the grid images ───────────────────────────────────────────────
# Two fixed three-quarter angles, each mesh fitted to the frame: every row is
# sized consistently and a re-run reproduces the same pictures.
#
# ZOOM is the only knob you normally touch -- it scales the auto-fit distance,
# so LOWER means CLOSER and less empty background:
#     1.05  default fit, generous margin
#     0.85  moderately tight
#     0.75  tight   <- current
#     0.65  very tight; a long/curved mesh may clip at the edges
ZOOM = 0.65

VIEWS = [dict(v, zoom=ZOOM) for v in kv.TIGHT_VIEWS]

# To use different angles, edit `dir` here (up is +y):
#   VIEWS = [{"dir": (1, 0.7, 1), "up": (0, 1, 0), "zoom": ZOOM},
#            {"dir": (-1, 0.5, -0.8), "up": (0, 1, 0), "zoom": ZOOM}]
# Or capture with your live viewer camera instead:  VIEWS = kv.DEFAULT_VIEWS
IMG_DIR    = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "images", "ph_images")
TAB_DIR    = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "tables")
# L-mesh exports land in notebooks/out/HL_results/[name]/exp_[tau]/, with the
# initial and optimised states side by side under that folder.
EXPORT_DIR = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "HL_results")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

# _signed_volume, load_oriented and ref_pair are defined in the setup cell.
def run_case(name, tau, data_dir=None):
    """Build + optimise one (mesh, tau) case.  Returns a record dict."""
    data_dir = data_dir or DATA
    V, F = load_oriented(name, data_dir)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )
    scaled_axes = axes / np.sin(angles[:, None])
    radii   = tau * smallest_radii
    if BD_RADII == "interior":
        radii = interior_mean_radii(basemesh, radii)
    elif BD_RADII == "zero":
        radii[basemesh.boundary_vertices()] = 0.0
    centers = basemesh.vertices + radii[:, None] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh, orientation=1, combinatorics="primal",
        mode="SphCongruence", radii=radii, centers=centers,
    )

    # notebooks/out/HL_results/[name]/exp_[tau][TAG]/
    exp_dir = os.path.join(EXPORT_DIR, name, f"exp_{tau:.2f}{TAG}")
    os.makedirs(exp_dir, exist_ok=True)

    # init_L_mesh() already ran compute_lmesh_structure(), so the initial state
    # is complete and exportable before any optimisation touches it.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "init")

    Vr, Fr = ref_pair(name, data_dir)
    with open(os.path.join(exp_dir, "reference_surface.obj"), "w") as fh:
        fh.writelines(f"v {x:.10g} {y:.10g} {z:.10g}\n" for x, y, z in Vr)
        fh.writelines(f"f {i+1} {j+1} {k+1}\n" for i, j, k in Fr)

    opt = setup_optimizer(lmesh, Vr, Fr, w_bd=W_BD if BD_FAIRNESS else 0.0)
    # Weights as registered, before the run (some terms damp theirs while optimising).
    weights = {k: float(w) for k, w in opt.get_term_weights_dict().items() if w != 0}

    for _ in range(N_STEPS):
        opt.get_gradients()
        opt.optimize_step()

    # Final total optimisation energy (weighted, as the optimiser sees it)
    X = opt.X
    total_e = float(sum(t.compute_energy(X) for t in opt.objective_terms.values()))

    lmesh.update_spheres_planes(
        opt.unpack("cv").reshape(-1, 3), opt.unpack("rv"),
        opt.unpack("npf").reshape(-1, 3), opt.unpack("hpf"),
    )

    # Oriented contact deviation of the optimised L-mesh.
    lmesh.improve_contact()
    dev = lmesh.compute_oriented_contact_dev()

    # improve_contact() ends with compute_lmesh_structure(), so the optimised
    # state is fully rebuilt here and safe to export.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "lmesh")
    print(f"    exported -> {os.path.relpath(exp_dir, PROJECT_ROOT)}")

    n_contact = len(lmesh.get_contact_points())

    return {
        "mesh": name, "tau": tau, "export_dir": exp_dir,
        "V": int(lmesh.top.V), "F": int(lmesh.top.F),
        "energy_total": total_e, "contact_dev": dev,
        "n_contact_points": n_contact,
        "weights": weights, "n_steps": N_STEPS,
        "bd_radii": BD_RADII, "bd_fairness": BD_FAIRNESS,
        "lmesh": lmesh,
    }


def latex_table(records, tau):
    """Render one tau's records as a LaTeX table (booktabs)."""
    def esc(s):
        return str(s).replace("_", r"\_")

    # Columns: mesh name (leftmost), |V|, |F|, total energy, contact deviation.
    lines = []
    lines.append("% Generated by hl_mesh_lab.ipynb -- tau = " + ("%.2f" % tau))
    lines.append(r"% Requires: \usepackage{booktabs}")
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"  \centering")
    lines.append(r"  \small")
    lines.append(r"  \begin{tabular}{lrrrr}")
    lines.append(r"    \toprule")
    head = ["Mesh", r"$|V|$", r"$|F|$", r"$E_{\mathrm{total}}$",
            r"$\delta_{\mathrm{contact}}$"]
    lines.append("    " + " & ".join(head) + r" \\")
    lines.append(r"    \midrule")
    for r in records:
        row = [esc(r["mesh"]), str(r["V"]), str(r["F"]),
               "%.3e" % r["energy_total"], "%.3e" % r["contact_dev"]]
        lines.append("    " + " & ".join(row) + r" \\")
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    # Steps and weights as recorded by run_case; every mesh shares one setup,
    # so a single list is printed unless they differ.
    n_steps = sorted({r["n_steps"] for r in records})
    w_sets  = [r["weights"] for r in records]
    def fmt_w(w):
        return ", ".join("\\texttt{" + esc(k) + "} $= " + ("%g" % v) + "$"
                         for k, v in w.items())
    if all(w == w_sets[0] for w in w_sets):
        w_txt = "Weights: " + fmt_w(w_sets[0]) + "."
    else:
        w_txt = "Weights: " + "; ".join(esc(r["mesh"]) + ": " + fmt_w(r["weights"])
                                         for r in records) + "."
    lines.append("  \\caption{HL-mesh optimisation after " +
                 "/".join(str(n) for n in n_steps) +
                 " steps, $\\tau = " + ("%.2f" % tau) + "$. " + w_txt + " "
                 "$\\delta_{\\mathrm{contact}}$ is the oriented contact "
                 "deviation after \\texttt{improve\\_contact()}.}")
    lines.append("  \\label{tab:hlmesh-tau-" + str(tau).replace(".", "p") + "}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# ── Sweep ─────────────────────────────────────────────────────────────────────
all_records = {}

for tau in TAUS:
    print(f"\n{'='*70}\n  tau = {tau}\n{'='*70}")
    records, grid_rows, grid_labels = [], [], []

    for name in ph_mesh_names:
        print(f"  [{name}]  tau={tau}")
        try:
            rec = run_case(name, tau)
        except Exception as e:
            print(f"    SKIP {type(e).__name__}: {e}")
            continue

        # Draw this case alone, then capture the two views.
        kv.clear()
        lmesh_visualize(rec["lmesh"])
        try:
            blobs = kv.screenshot_views(views=VIEWS, width=640, height=480)
            for i, b in enumerate(blobs):
                fn = os.path.join(IMG_DIR, f"{name}_{tau:.2f}{TAG}_v{i}.png")
                with open(fn, "wb") as fh:
                    fh.write(b)
            grid_rows.append(blobs)
            grid_labels.append(f"{name}   tau={tau:.2f}   "
                               f"E={rec['energy_total']:.2e}   "
                               f"dev={rec['contact_dev']:.2e}")
        except Exception as e:
            print(f"    (no screenshot: {e})")

        rec.pop("lmesh", None)          # don't keep meshes alive in the record
        records.append(rec)
        print(f"    E_total={rec['energy_total']:.4e}   "
              f"contact_dev={rec['contact_dev']:.4e}   "
              f"contact_pts={rec['n_contact_points']}")

    all_records[tau] = records

    # LaTeX table for this tau
    if records:
        tex  = latex_table(records, tau)
        path = os.path.join(TAB_DIR, f"ph_energy_tau{tau:.2f}{TAG}.tex")
        with open(path, "w") as fh:
            fh.write(tex + "\n")
        print(f"\n  -> wrote {path}")
        print(tex)

    # Contact-sheet grid for this tau
    if grid_rows:
        gpath = os.path.join(IMG_DIR, f"grid_tau{tau:.2f}{TAG}.png")
        kv.save_grid(gpath, grid_rows, labels=grid_labels)
        print(f"  -> wrote {gpath}")

# Raw numbers, in case you want to re-render the tables differently
with open(os.path.join(TAB_DIR, f"ph_sweep_results{TAG}.json"), "w") as fh:
    json.dump(all_records, fh, indent=2)
print(f"\nDone. Tables in {TAB_DIR}, images in {IMG_DIR}")



  tau = 0.5
  [bumpy_patch.out_d_p]  tau=0.5
Mesh |V|=681, |F|=306, |E|=986
Mesh |V|=681, |F|=306, |E|=986
Thread pool: 6 workers (terms=6, cpus=32)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 1.863802e-03  | 4.103056e-07  | 1.670064e+00  | 4.083177e-05  | N/A      | 0.5757  
10     | 8.937305e-07  | 3.008380e-10  | 3.699750e-02  | 4.083177e-05  | N/A      | 0.0951  
20     | 2.156498e-07  | 7.801223e-11  | 1.331092e-02  | 4.083177e-05  | N/A      | 0.0814  
L-mesh Optimized
Oriented Contact Deviation: 4.748e-11
    exported -> notebooks/out/lmesh/HL_results/bumpy_patch.out_d_p/exp_0.50
    (no screenshot: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.)
    E_total=1.9821e-07   contact_dev=1.2110e-29   contact_pts=1837
  [coiled_torus_segment.ou